In [1]:
import os
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [2]:
TRAIN_DIR = "../data/phone_detection/split/train"
VAL_DIR = "../data/phone_detection/split/val"
TEST_DIR = "../data/phone_detection/split/test"
MODEL_DIR = "../models/phone_detection"

os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-5

EPOCHS_STAGE1 = 8
EPOCHS_STAGE2 = 4

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    

Found 1656 files belonging to 3 classes.
Found 355 files belonging to 3 classes.
Found 360 files belonging to 3 classes.
Classes: ['multiple_phones', 'no_phone', 'phone']
Number of classes: 3


In [4]:
with open(os.path.join(MODEL_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f)

print("Class names saved.")

Class names saved.


In [5]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [6]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.1, 0.1),
    ],
    name="data_augmentation"
)

In [7]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)

base_model.trainable = False

print("EfficientNetB0 base model loaded.")

EfficientNetB0 base model loaded.


In [8]:
import tensorflow as tf

print("GPUs:", tf.config.list_physical_devices('GPU'))

GPUs: []


In [9]:
inputs = keras.Input(shape=IMG_SIZE + (3,), name="input_image")

x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.Dropout(0.4, name="dropout")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

model = keras.Model(inputs, outputs, name="phone_detection_classifier")

model.summary()

Model: "phone_detection_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_avg_pool                 │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,053,414 (15.46 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [10]:
checkpoint_path = os.path.join(MODEL_DIR, "best_phone_detection.keras")

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1, min_lr=1e-6)
]

print("Callbacks ready.")

Callbacks ready.


In [11]:
from PIL import Image
import os

ROOT_DIR = "../data/phone_detection/split"
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

removed = 0

for root, dirs, files in os.walk(ROOT_DIR):
    for file in files:
        if file.lower().endswith(VALID_EXTS):
            path = os.path.join(root, file)
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception:
                os.remove(path)
                removed += 1
                print("Removed corrupted image:", path)

print(f"\nTotal corrupted images removed: {removed}")


Total corrupted images removed: 0


In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks
)

Epoch 1/8
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.6561 - loss: 0.7774
Epoch 1: val_loss improved from None to 0.38143, saving model to ../models/phone_detection\best_phone_detection.keras

Epoch 1: finished saving model to ../models/phone_detection\best_phone_detection.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 30s 215ms/step - accuracy: 0.7687 - loss: 0.5869 - val_accuracy: 0.8704 - val_loss: 0.3814 - learning_rate: 0.0010
Epoch 2/8
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.8586 - loss: 0.3764
Epoch 2: val_loss improved from 0.38143 to 0.33287, saving model to ../models/phone_detection\best_phone_detection.keras

Epoch 2: finished saving model to ../models/phone_detection\best_phone_detection.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 20s 193ms/step - accuracy: 0.8768 - loss: 0.3339 - val_accuracy: 0.8761 - val_loss: 0.3329 - learning_rate: 0.0010
Epoch 3/8
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.8806 - loss: 0.3160
Epoch 3: val_loss improved from 0.33

In [13]:
base_model.trainable = True

fine_tune_at = len(base_model.layers) - 20

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks
)

Epoch 1/4
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.7981 - loss: 0.5551
Epoch 1: val_loss did not improve from 0.25982
104/104 ━━━━━━━━━━━━━━━━━━━━ 33s 226ms/step - accuracy: 0.7941 - loss: 0.5435 - val_accuracy: 0.9155 - val_loss: 0.2924 - learning_rate: 1.0000e-05
Epoch 2/4
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step - accuracy: 0.8499 - loss: 0.4226
Epoch 2: val_loss did not improve from 0.25982

Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
104/104 ━━━━━━━━━━━━━━━━━━━━ 22s 214ms/step - accuracy: 0.8442 - loss: 0.4242 - val_accuracy: 0.9070 - val_loss: 0.3136 - learning_rate: 1.0000e-05
Epoch 3/4
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.8647 - loss: 0.3860
Epoch 3: val_loss did not improve from 0.25982
104/104 ━━━━━━━━━━━━━━━━━━━━ 22s 211ms/step - accuracy: 0.8521 - loss: 0.4006 - val_accuracy: 0.9042 - val_loss: 0.3188 - learning_rate: 5.0000e-06
Epoch 3: early stopping
Restoring model weights from the end of the best epoc

In [14]:
final_model_path = os.path.join(MODEL_DIR, "final_phone_detection.keras")
model.save(final_model_path)

print("Final model saved to:", final_model_path)

Final model saved to: ../models/phone_detection\final_phone_detection.keras


In [15]:
test_loss, test_acc = model.evaluate(test_ds)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 139ms/step - accuracy: 0.9000 - loss: 0.2915
Test Loss: 0.2915
Test Accuracy: 0.9000


In [16]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

all_labels = []
all_preds = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_labels.extend(np.argmax(labels.numpy(), axis=1))

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=class_names)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(report)

Confusion Matrix:
[[ 58   0  15]
 [  3  77   5]
 [  8   5 189]]

Classification Report:
                 precision    recall  f1-score   support

multiple_phones       0.84      0.79      0.82        73
       no_phone       0.94      0.91      0.92        85
          phone       0.90      0.94      0.92       202

       accuracy                           0.90       360
      macro avg       0.89      0.88      0.89       360
   weighted avg       0.90      0.90      0.90       360



In [ ]:
import numpy as np

def predict_image(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = tf.keras.applications.efficientnet.preprocess_input(arr)

    preds = model.predict(arr, verbose=0)
    pred_idx = int(np.argmax(preds[0]))
    pred_class = class_names[pred_idx]
    confidence = float(np.max(preds[0]))

    return {
        "predicted_class": pred_class,
        "confidence": confidence
    }